# M3–M5 risk, optimizer and benchmark backtest
Run each stage separately so any error is visible. Missing baseline outputs are rebuilt automatically.

In [ ]:
# Bootstrap this notebook even when it is opened in a fresh Colab kernel.
from pathlib import Path
import urllib.request
_BOOTSTRAP_REPO = Path('/content/kltn')
_BOOTSTRAP_SCRIPT = Path('/content/colab_bootstrap.py')
raw = 'https://raw.githubusercontent.com/maiphuowng205/kltn/fcb0507351d694c2431e454bd84a357958f96635/scripts/colab_bootstrap.py'
urllib.request.urlretrieve(raw, str(_BOOTSTRAP_SCRIPT))
exec(_BOOTSTRAP_SCRIPT.read_text(encoding='utf-8'), globals())


In [ ]:
# Ensure the forecast artifact required by the portfolio stage exists.
import subprocess, sys
baseline_run = WORKSPACE / 'runs' / 'v3_forecast_baselines'
def run_stage(label, script_name, extra_args):
    script_path = Path(script_name)
    if not script_path.is_absolute(): script_path = REPO / 'scripts' / script_path
    cmd = [sys.executable, str(script_path), *extra_args]
    print(f'\n=== {label} ===')
    print(' '.join(map(str, cmd)))
    result = subprocess.run(cmd, check=False, capture_output=True, text=True)
    if result.stdout: print(result.stdout)
    if result.stderr: print('STDERR:\n' + result.stderr)
    print(f'[{label}] returncode={result.returncode}')
    if result.returncode != 0:
        detail = (result.stderr or result.stdout or 'no subprocess output').strip()
        raise RuntimeError(f'{label} failed with returncode {result.returncode}:\n{detail}')
    return result
if not (baseline_run / 'forecasts.parquet').exists():
    run_stage('forecast baselines (auto-rebuild)', 'run_v3_forecast_baselines.py', ['--data-root', str(DATA_ROOT), '--run-dir', str(baseline_run)])
else:
    print('Existing forecast baseline found:', baseline_run / 'forecasts.parquet')


In [ ]:
# Risk coverage: train/validation covariance history only.
for required in [DATA_ROOT / 'curated/universe_weekly.parquet', DATA_ROOT / 'curated/daily_panel.parquet']:
    print('input', required, 'exists=', required.exists(), 'bytes=', required.stat().st_size if required.exists() else None)
risk_run = WORKSPACE / 'runs' / 'v3_risk_coverage'
risk_script = Path('/content/run_v3_risk_coverage_fixed.py')
if not risk_script.exists():
    raw = 'https://raw.githubusercontent.com/maiphuowng205/kltn/8f38b06/scripts/run_v3_risk_coverage.py'
    urllib.request.urlretrieve(raw, risk_script)
run_stage('risk coverage', str(risk_script), ['--data-root', str(DATA_ROOT), '--run-dir', str(risk_run)])
import pandas as pd
pd.read_parquet(risk_run / 'risk_coverage_summary.parquet')


In [ ]:
# Portfolio benchmark ladder and cost-aware optimizer.
portfolio_run = WORKSPACE / 'runs' / 'v3_portfolio_benchmarks'
run_stage('portfolio benchmarks', 'run_v3_portfolio_benchmarks.py', ['--data-root', str(DATA_ROOT), '--forecast-run', str(baseline_run), '--run-dir', str(portfolio_run)])
pd.read_parquet(portfolio_run / 'portfolio_metrics_summary.parquet')


In [ ]:
# Determinism reruns for the baseline and portfolio artifacts.
baseline_repeat = WORKSPACE / 'runs' / 'v3_forecast_baselines_repeat'
portfolio_repeat = WORKSPACE / 'runs' / 'v3_portfolio_benchmarks_repeat'
run_stage('forecast baseline repeat', 'run_v3_forecast_baselines.py', ['--data-root', str(DATA_ROOT), '--run-dir', str(baseline_repeat)])
run_stage('portfolio repeat', 'run_v3_portfolio_benchmarks.py', ['--data-root', str(DATA_ROOT), '--forecast-run', str(baseline_repeat), '--run-dir', str(portfolio_repeat)])
run_stage('forecast determinism check', 'validate_v3_determinism.py', ['--main-run', str(baseline_run), '--repeat-run', str(baseline_repeat), '--files', 'forecasts.parquet', 'forecast_metrics_by_date.parquet', 'forecast_metrics_summary.parquet', '--report', str(WORKSPACE / 'runs' / 'v3_determinism_forecast.json')])
run_stage('portfolio determinism check', 'validate_v3_determinism.py', ['--main-run', str(portfolio_run), '--repeat-run', str(portfolio_repeat), '--files', 'portfolio_returns.parquet', 'weights.parquet', 'trades.parquet', 'solver_log.parquet', 'portfolio_metrics_summary.parquet', '--report', str(WORKSPACE / 'runs' / 'v3_determinism_portfolio.json')])
import json
determinism = {'forecast': json.loads((WORKSPACE / 'runs' / 'v3_determinism_forecast.json').read_text()), 'portfolio': json.loads((WORKSPACE / 'runs' / 'v3_determinism_portfolio.json').read_text())}
(WORKSPACE / 'runs' / 'v3_determinism_report.json').write_text(json.dumps(determinism, indent=2))
print(json.dumps(determinism, indent=2))


In [ ]:
# Persist completed M3–M5 artifacts to Drive for later notebooks/runtimes.
from pathlib import Path
from shutil import copytree, copy2
drive_runs = Path('/content/drive/MyDrive/kltn/runs')
drive_runs.mkdir(parents=True, exist_ok=True)
for name in ['v3_forecast_baselines', 'v3_risk_coverage', 'v3_portfolio_benchmarks']:
    source = WORKSPACE / 'runs' / name
    if source.exists(): copytree(source, drive_runs / name, dirs_exist_ok=True)
for name in ['v3_determinism_forecast.json', 'v3_determinism_portfolio.json', 'v3_determinism_report.json']:
    source = WORKSPACE / 'runs' / name
    if source.exists(): copy2(source, drive_runs / name)
print('M3–M5 artifacts synced to:', drive_runs)
